# Data Gate — Uzbek Car Recognizer

Reproducible data pipeline: **scraped listing photos → clean, leakage-safe train/val/test split**.

**Run top to bottom.** Re-run the whole notebook whenever you replace the dataset zip on
Drive (e.g. after adding more Damas). Runtime: set **Runtime → Change runtime type → T4 GPU**.

Pipeline: `unzip → YOLO crop+filter → CLIP label-cleaning → split by listing`.

In [ ]:
# ── Cell 1 · Setup ──────────────────────────────────────────────
!pip install -q ultralytics transformers ftfy
import torch, torch.nn.functional as F, numpy as np, pandas as pd
import random, shutil, glob
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", dev)

In [ ]:
# ── Cell 2 · Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3 · Unzip dataset + class counts ───────────────────────
# Loads the clean dataset zip from Drive (CapstoneCars/). The rm -rf makes re-runs clean
# after you replace the zip.
!rm -rf /content/data
zips = [z for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
        if any(k in z.lower() for k in ('car','dataset','clean','raw'))]
print("Found zips:", zips)
ZIP = zips[0]                        # if more than one, set the right path manually
!mkdir -p /content/data && unzip -q "$ZIP" -d /content/data

CLASSES = ['cobalt','nexia3','spark','gentra','damas']
base = next(p.parent for p in Path('/content/data').rglob('cobalt') if p.is_dir())
print("dataset base =", base)
for c in CLASSES:
    print(f"  {c:8} {len(list((base/c).glob('*')))}")

In [ ]:
# ── Cell 4 · YOLO crop + exterior/interior filter ───────────────
# Pretrained detector finds the car, crops to it, and DROPS photos with no car
# (interiors / engine / documents / stock). Data prep — NOT the graded model.
from ultralytics import YOLO
VEHICLES, CONF, PAD, AREA_FLOOR = [2, 5, 7], 0.25, 0.08, 0.03   # COCO car/bus/truck
DST = Path('/content/crops'); shutil.rmtree(DST, ignore_errors=True)
det = YOLO('yolo11s.pt')
rows = []
for cls in CLASSES:
    files = [p for p in (base/cls).iterdir()
             if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
    out = DST/cls; out.mkdir(parents=True, exist_ok=True); kept = 0
    for p in tqdm(files, desc=cls):
        try: im = Image.open(p).convert('RGB')
        except Exception: rows.append([cls, p.name, 0, 'unreadable']); continue
        W, H = im.size
        r = det.predict(str(p), classes=VEHICLES, conf=CONF, imgsz=640,
                        device=0, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            rows.append([cls, p.name, 0, 'no_car']); continue
        b = r.boxes.xyxy.cpu().numpy(); a = (b[:,2]-b[:,0])*(b[:,3]-b[:,1]); i = a.argmax()
        if a[i] < AREA_FLOOR*W*H:
            rows.append([cls, p.name, 0, 'car_too_small']); continue
        x1, y1, x2, y2 = b[i]; bw, bh = x2-x1, y2-y1
        box = (max(0, int(x1-PAD*bw)), max(0, int(y1-PAD*bh)),
               min(W, int(x2+PAD*bw)), min(H, int(y2+PAD*bh)))
        im.crop(box).save(out/f"{p.stem}.jpg", quality=92)
        rows.append([cls, p.name, 1, 'ok']); kept += 1
    print(f"{cls}: kept {kept}/{len(files)}")
pd.DataFrame(rows, columns=['class','file','kept','reason']).to_csv(
    '/content/crops_manifest.csv', index=False)
print("CROP DONE")

In [ ]:
# ── Cell 5 · CLIP scores: interior + wrong-model ────────────────
# CLIP = image/text embeddings. Two signals per crop:
#   interior : looks "inside a car" vs "outside" (zero-shot text match)
#   agree    : fraction of 10 nearest neighbors sharing its label (low = wrong model)
from transformers import CLIPModel, CLIPProcessor
from sklearn.neighbors import NearestNeighbors
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(dev).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

@torch.no_grad()
def txt_emb(tok):
    o = clip.text_model(**tok); return F.normalize(clip.text_projection(o.pooler_output), dim=-1)
@torch.no_grad()
def img_emb(px):
    o = clip.vision_model(**px); return F.normalize(clip.visual_projection(o.pooler_output), dim=-1)

CROPS = Path('/content/crops')
paths  = [p for c in CLASSES for p in (CROPS/c).glob('*.jpg')]
labels = np.array([p.parent.name for p in paths])
prompts = ["a photo of a car from outside, exterior view",
           "the inside of a car: dashboard, steering wheel, seats"]
tfeat = txt_emb(proc(text=prompts, return_tensors="pt", padding=True).to(dev))

embs, interior = [], []
for i in tqdm(range(0, len(paths), 64)):
    imgs = [Image.open(p).convert('RGB') for p in paths[i:i+64]]
    f = img_emb(proc(images=imgs, return_tensors="pt").to(dev))
    sim = f @ tfeat.T
    embs.append(f.cpu()); interior.extend((sim[:,1]-sim[:,0]).cpu().numpy())
embs = torch.cat(embs).numpy(); interior = np.array(interior)

_, idx = NearestNeighbors(n_neighbors=11, metric='cosine').fit(embs).kneighbors(embs)
agree = np.array([(labels[idx[i,1:]] == labels[i]).mean() for i in range(len(labels))])
pd.DataFrame({'path': [str(p) for p in paths], 'label': labels,
              'interior': interior, 'agree': agree}).to_csv('/content/clean_scores.csv', index=False)
print("CLIP scores done for", len(paths), "crops")

In [ ]:
# ── Cell 6 · Clean + leakage-safe split by listing ──────────────
df = pd.read_csv('/content/clean_scores.csv')
INT_THRESH, AGREE_THRESH = 0.0, 0.15        # conservative on wrong-model to keep look-alikes
df['reason'] = 'keep'
df.loc[df.agree < AGREE_THRESH,  'reason'] = 'wrong_model'
df.loc[df.interior > INT_THRESH, 'reason'] = 'interior'
print("Removed per class × reason:\n", df[df.reason!='keep'].groupby(['label','reason']).size())
keep = df[df.reason=='keep'].copy()

# filenames are <hash>_<LISTINGID>_<n>.jpg  → the listing id is the MIDDLE field
def listing_of(path):
    parts = Path(path).stem.split('_'); return parts[1] if len(parts) >= 3 else parts[0]
keep['file']    = keep.path.apply(lambda p: Path(p).name)
keep['listing'] = keep.path.apply(listing_of)
keep['class']   = keep.label
print(f"\n{len(keep)} clean crops · {keep.listing.nunique()} listings "
      f"(avg {len(keep)/keep.listing.nunique():.1f} photos/listing)")

# assign whole LISTINGS to a split (70/15/15), stratified per class → no leakage
random.seed(42)
lst = keep.groupby('listing')['class'].agg(lambda s: s.mode().iat[0]).reset_index()
split_of = {}
for c, g in lst.groupby('class'):
    ids = sorted(g.listing); random.shuffle(ids)
    n = len(ids); ntr, nva = int(n*0.70), int(n*0.15)
    for i, lid in enumerate(ids):
        split_of[lid] = 'train' if i < ntr else ('val' if i < ntr+nva else 'test')
keep['split'] = keep.listing.map(split_of)

OUT = Path('/content/dataset'); shutil.rmtree(OUT, ignore_errors=True)   # clear old split
for _, r in keep.iterrows():
    d = OUT/r.split/r['class']; d.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.path, d/r.file)
keep.to_csv('/content/split_manifest.csv', index=False)

print("\nImages per split × class:\n",
      keep.groupby(['split','class']).size().unstack(fill_value=0).loc[['train','val','test']])
print("\nListings per split × class:\n",
      keep.groupby(['split','class'])['listing'].nunique().unstack(fill_value=0).loc[['train','val','test']])
leaked = int((keep.groupby('listing')['split'].nunique() > 1).sum())
print(f"\nLEAKAGE CHECK — listings in >1 split: {leaked}")
assert leaked == 0, "LEAKAGE!"
print("✅ leakage-safe split ready")

In [ ]:
# ── Cell 7 · Back up the split to Drive ─────────────────────────
!cd /content && zip -q -r dataset_split.zip dataset split_manifest.csv >/dev/null
!cp /content/dataset_split.zip /content/drive/MyDrive/CapstoneCars/
print("✅ split + manifest backed up to Drive")

In [ ]:
# ── Cell 8 · (optional) Visual sanity check ─────────────────────
import matplotlib.pyplot as plt
km = pd.read_csv('/content/split_manifest.csv')
fig, ax = plt.subplots(len(CLASSES), 6, figsize=(14, 2.3*len(CLASSES)))
for row, c in enumerate(CLASSES):
    fs = km[km['class'] == c]['path'].tolist(); random.shuffle(fs)
    for k in range(6):
        ax[row][k].axis('off')
        if k < len(fs): ax[row][k].imshow(Image.open(fs[k]))
    ax[row][0].set_title(c, loc='left')
plt.suptitle("Final clean crops per class"); plt.tight_layout(); plt.show()